# NYXVAULT — запуск по отдельным шагам\n\nЭтот вариант вместо одной длинной ячейки разделяет процесс на 4 шага. Так видно, где именно возникла задержка.\n\n**Перед запуском:** в старом Colab открой меню «Среда выполнения» → «Перезапустить сеанс». Это остановит зависшие ячейки. Затем запускай эти ячейки строго по одной сверху вниз. Не запускай следующую, пока текущая не завершилась.\n\nМодель загружается в Colab, не на телефон. Туннель временный: сессия Colab должна оставаться подключённой.

In [ ]:
# ШАГ 1. Загружаем только серверный код из GitHub.import requestsfrom pathlib import PathSERVER_DIR = Path("/content/nyxvault_server")SERVER_DIR.mkdir(parents=True, exist_ok=True)BASE = "https://raw.githubusercontent.com/Phantom220-hub/Nyxvault/nyxvault-cpu-server/server/"for filename in ("app.py", "requirements.txt"):    print(f"Загружаю {filename}...", flush=True)    response = requests.get(BASE + filename, timeout=(15, 60))    response.raise_for_status()    target = SERVER_DIR / filename    target.write_bytes(response.content)    print(f"Готово: {filename}, {target.stat().st_size:,} байт", flush=True)print("ШАГ 1 ГОТОВ. Файлы модели пока не загружались.", flush=True)

In [ ]:
# ШАГ 2. Устанавливаем библиотеки. Здесь pip будет показывать ход работы.import sysimport subprocessprint("Установка зависимостей NYXVAULT...", flush=True)subprocess.run(    [sys.executable, "-m", "pip", "install", "-r", "/content/nyxvault_server/requirements.txt"],    check=True)print("ШАГ 2 ГОТОВ.", flush=True)

In [ ]:
# ШАГ 3. Запускаем настоящий API и показываем новые строки логов по мере появления.import osimport sysimport timeimport secretsimport subprocessimport requestsfrom pathlib import PathSERVER_DIR = Path("/content/nyxvault_server")LOG_DIR = Path("/content/nyxvault_server_logs")LOG_DIR.mkdir(parents=True, exist_ok=True)API_TOKEN = secrets.token_urlsafe(32)os.environ["NYXVAULT_API_TOKEN"] = API_TOKENos.environ["NYXVAULT_MODEL_DIR"] = "/content/nyxvault_model"headers = {"Authorization": f"Bearer {API_TOKEN}"}api_log_path = LOG_DIR / "api.log"api_log_file = open(api_log_path, "w", buffering=1)server = subprocess.Popen(    [sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],    cwd=str(SERVER_DIR),    env=os.environ.copy(),    stdout=api_log_file,    stderr=subprocess.STDOUT,    start_new_session=True,)print("Сервер запущен. Жду загрузку модели; новые логи будут показаны ниже.", flush=True)deadline = time.time() + 600shown = 0ready = Falsewhile time.time() < deadline:    time.sleep(4)    if api_log_path.exists():        log_text = api_log_path.read_text(errors="replace")        if len(log_text) > shown:            print(log_text[shown:], end="", flush=True)            shown = len(log_text)    if server.poll() is not None:        raise RuntimeError("Процесс API завершился. Последние строки лога:\n" + api_log_path.read_text(errors="replace")[-12000:])    try:        check = requests.get("http://127.0.0.1:8000/health", headers=headers, timeout=5)        if check.ok:            print("API ГОТОВ:", check.json(), flush=True)            ready = True            break    except requests.RequestException:        passif not ready:    raise TimeoutError("За 10 минут модель не загрузилась. Последние строки лога:\n" + api_log_path.read_text(errors="replace")[-12000:])print("ШАГ 3 ГОТОВ. API_TOKEN сохранён только в переменной этой сессии.", flush=True)

In [ ]:
# ШАГ 4. Создаём временный HTTPS-адрес и проверяем генерацию через него.import osimport reimport platformimport subprocessimport timeimport requestsfrom pathlib import Pathmachine = platform.machine().lower()arch = "arm64" if machine in ("aarch64", "arm64") else "amd64"cloudflared = Path("/content/cloudflared")if not cloudflared.exists():    url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"    print("Загружаю cloudflared:", arch, flush=True)    with requests.get(url, stream=True, timeout=(15, 120)) as response:        response.raise_for_status()        with open(cloudflared, "wb") as output:            total = 0            for chunk in response.iter_content(chunk_size=1024 * 1024):                if chunk:                    output.write(chunk)                    total += len(chunk)                    print(f"Получено {total / (1024 * 1024):.1f} MB", flush=True)    cloudflared.chmod(0o755)tunnel_log_path = Path("/content/nyxvault_server_logs/tunnel.log")tunnel_log_file = open(tunnel_log_path, "w", buffering=1)tunnel = subprocess.Popen(    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],    stdout=tunnel_log_file,    stderr=subprocess.STDOUT,    start_new_session=True,)print("Запускаю HTTPS-туннель...", flush=True)public_url = Nonedeadline = time.time() + 90shown = 0while time.time() < deadline:    time.sleep(2)    log_text = tunnel_log_path.read_text(errors="replace") if tunnel_log_path.exists() else ""    if len(log_text) > shown:        print(log_text[shown:], end="", flush=True)        shown = len(log_text)    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)    if match:        public_url = match.group(0)        break    if tunnel.poll() is not None:        raise RuntimeError("Cloudflare tunnel завершился:\n" + log_text[-8000:])if not public_url:    raise TimeoutError("Не удалось получить HTTPS-адрес. Лог:\n" + tunnel_log_path.read_text(errors="replace")[-8000:])test_health = requests.get(public_url + "/health", headers={"Authorization": f"Bearer {API_TOKEN}"}, timeout=30)test_health.raise_for_status()test_generation = requests.post(    public_url + "/generate",    headers={"Authorization": f"Bearer {API_TOKEN}"},    json={"prompt": "Пользователь: Привет.\nNYXVAULT:", "max_new_tokens": 12, "temperature": 0.0, "top_k": 40},    timeout=180)test_generation.raise_for_status()print("\nСЕРВЕР ПРОШЁЛ ПРОВЕРКУ")print("NYXVAULT_BACKEND_URL =", public_url)print("NYXVAULT_BACKEND_TOKEN =", API_TOKEN)print("Тест генерации:", test_generation.json())print("Скопируй URL и TOKEN в Secrets проекта Lovable. TOKEN никому не отправляй и не публикуй.")